# Class 07 · Manipulating and Visualizing Data (Python)

Live-coding companion to the Week 4, Class A slides. Run the cells from top to bottom; each step builds on the previous one and ends with a saved figure, its summary table and the record of excluded rows.

All player, hour and achievement values are fictional.

In [ ]:
%matplotlib inline

## Part 1 · Tables and the meaning of a row

### Start with the meaning of a row

A table operation changes the available evidence. Our fictional game example has one record per player–game pair. A second table supplies a genre and an achievement total for each listed game.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris

> Run the examples in order, in a folder where output files can be saved.

### A tibble extends a data frame

- A tibble inherits from `data.frame`, with its own printing and subsetting.
- `tb[, "x"]` keeps a table; `tb[["x"]]` extracts a vector.
- In pandas, `tb[["x"]]` is a DataFrame and `tb["x"]` is a Series.

In [ ]:
tb = pd.DataFrame({"x": [1, 2, 3],
                   "y": ["a", "b", "c"]})
print(type(tb[["x"]]))  # DataFrame, a table
print(type(tb["x"]))  # Series, labelled column

> Ask: which of these is still a table?

### Teaching data: eight records and a game lookup

- Player names, hours, achievements and lookup values are fictional.
- Dan has one unknown hours value.
- Tunic appears in the play records but has no lookup entry.
- Stardew appears only in the lookup table.

Keep the original tables. Derived tables get new names.

In [ ]:
# Fictional player-game records for teaching.
plays = pd.DataFrame({
    "player": ["ana", "ana", "ben", "ben",
               "cara", "cara", "dan", "dan"],
    "game": ["Portal", "Celeste", "Portal",
             "Hades", "Celeste", "Hades",
             "Portal", "Tunic"],
    "hours": [12.5, 30, 8, 45.5, 22, 17.5,
              np.nan, 6],
    "achieved": [9, 14, 4, 25, 11, 8, 2, 3]})

In [ ]:
# Fictional lookup values, not official stats.
games = pd.DataFrame({
    "game": ["Portal", "Celeste", "Hades",
             "Stardew"],
    "genre": ["puzzle", "platformer",
              "roguelike", "farming"],
    "total": [15, 30, 49, 40]})
print(plays)
print(games)

> Each game key is unique in the lookup.

## Part 2 · Six table operations

### Select records

Conditions in `filter` are combined with AND; pandas keeps the original index labels.

In [ ]:
print(plays.loc[plays["hours"] > 20])
print(plays.query(
    "hours > 20 and player != 'ana'"))
print(plays.loc[plays["game"].isin(
    ["Hades", "Tunic"])])

> Predict first: which rows have more than 20 hours? (Ana–Celeste, Ben–Hades, Cara–Celeste)

### Make the missing-value rule explicit

- R comparisons with `NA` give an unknown result; `filter` keeps only `TRUE`.
- For the float `NaN` in pandas, `NaN > 20` is false but `NaN != 20` is true.
- Write the policy into the code instead of inheriting a default.

In [ ]:
print(len(plays.loc[plays["hours"] > 20]))   # 3
print(len(plays.loc[plays["hours"] != 20]))  # 8
known = plays.loc[
    plays["hours"].notna()
    & (plays["hours"] != 20)]
print(len(known))  # 7, explicit NA policy

> The explicit known-hours selection gives seven records in both languages.

### Select and rename variables

A label range includes both endpoints. An ordinary Python positional slice excludes its stop.

In [ ]:
print(plays[["player", "game", "hours"]])
print(plays.drop(columns="achieved"))
print(plays.loc[:, "player":"hours"])
print(plays.select_dtypes("number"))
print(plays.rename(columns={"hours": "hrs"}))

### Order records

Missing hours appear last. Add ordering columns when ties matter.

In [ ]:
print(plays.sort_values(
    "hours", ascending=False,
    kind="stable", na_position="last"))
print(plays.sort_values(
    ["player", "achieved"],
    ascending=[True, False]))

### Derive a rate

Rate = achievements per known positive hour. Missing or non-positive hours give a missing rate and band.

In [ ]:
rated = plays.assign(
    denominator=lambda d: d.hours.where(
        d.hours > 0),
    rate=lambda d: d.achieved / d.denominator,
    band=lambda d: pd.Series(
        np.where(d.rate > 0.5, "hi", "lo"),
        index=d.index, dtype="string",
    ).where(d.rate.notna()))
print(rated[["player", "game", "rate", "band"]])

> Ask: what should Dan–Portal's rate be?

### Report the summary denominator

Say how many values a summary uses.

In [ ]:
print(plays["hours"].mean(skipna=False))  # nan
overview = pd.DataFrame([{
    "records": len(plays),
    "observed_hours": plays["hours"].count(),
    "mean_hours":
        plays["hours"].mean(skipna=True),
    "achievements": plays["achieved"].sum()}])
print(overview)
# 8 records, 7 observed, 20.214286 h, 76

> Four players, eight records, seven observed hours; the mean describes those seven.

## Part 3 · Grouped calculations

### Summarize each player

Choose groups, calculate one summary per group, then drop the grouping. In R, `x |> f()` passes `x` as the first argument of `f`; a pandas method chain passes the table to the next method.

In [ ]:
by_player = (plays
    .groupby("player", as_index=False)
    .agg(records=("game", "size"),
         observed_hours=("hours", "count"),
         known_hours=("hours", "sum"),
         achievements=("achieved", "sum")))
print(by_player)

> Dan's known hours (6) are a partial observed sum: his Portal hours are unknown.

### Keep the original record count

A grouped mutate or transform keeps every record; a summary collapses them.

In [ ]:
shares = plays.assign(share_known_hours=(
    plays.hours
    / plays.groupby("player").hours
      .transform("sum")))
print(shares[["player", "game",
              "share_known_hours"]])

> Eight records come back, not four.

### Control group order

In [ ]:
dan_first = plays.sort_values(
    "player", ascending=False)
print(dan_first.groupby("player", sort=False,
                        as_index=False)
      .agg(achievements=("achieved", "sum")))
# First appearance: dan, cara, ben, ana.

> First-appearance groups: dan, cara, ben, ana. Default grouped summaries sort keys.

### Retain an unobserved category

Eve is a possible category with no recorded rows. Absence from these records does not prove zero activity.

In [ ]:
levels_player = ["ana", "ben", "cara", "dan",
                 "eve"]
p = plays.assign(player=pd.Categorical(
    plays.player, categories=levels_player))
empty_groups = (p
    .groupby("player", observed=False,
             as_index=False)
    .agg(records=("game", "size")))
print(empty_groups)  # eve has 0 recorded rows

### Compose the six operations

The mean gives every eligible record equal weight. It is not total achievements divided by total hours.

In [ ]:
player_rates = (plays
    .loc[plays.hours.notna()
         & (plays.hours > 0)]
    .assign(rate=lambda d: d.achieved / d.hours)
    .groupby("player", as_index=False)
    .agg(records=("game", "size"),
         mean_rate=("rate", "mean"))
    .sort_values("mean_rate", ascending=False)
    [["player", "records", "mean_rate"]])
print(player_rates)

## Part 4 · Combining tables

### Append compatible records

Appending rows aligns column names. Appending does not change the original lookup.

In [ ]:
extra = pd.DataFrame({"game": ["Tunic"],
    "genre": ["puzzle"], "total": [42]})
extended = pd.concat([games, extra],
                     ignore_index=True)
print(extended)  # games itself still has 4 rows
a = pd.DataFrame({"x": [1]})
b = pd.DataFrame({"y": [2]})
print(pd.concat([a, b], ignore_index=True))

> Check column names, units and types before binding.

### Check row correspondence

Adding columns by position needs an established correspondence between rows.

In [ ]:
x = pd.DataFrame({"id": ["a", "b"],
                  "value": [2, 3]})
y = pd.DataFrame({"id": ["a", "b"],
                  "label": ["low", "high"]})
assert x["id"].equals(y["id"])
print(pd.concat([x, y[["label"]]], axis=1))

### Validate the lookup relationship

Check complete keys and a unique lookup key before the join. Duplicate lookup keys would create extra matches.

In [ ]:
assert plays["game"].notna().all()
assert (games["game"].notna().all()
        and games["game"].is_unique)
joined = plays.merge(games, on="game",
                     how="left",
                     validate="many_to_one")
assert len(joined) == len(plays)
print(joined)

> The left join keeps all eight play records.

### Compare the four join counts

In [ ]:
print(len(joined))                  # 8
print(len(plays.merge(games, on="game",
                      how="inner")))  # 7
print(len(plays.merge(games, on="game",
                      how="right")))  # 8
print(len(plays.merge(games, on="game",
                      how="outer")))  # 9

> Predict first: inner 7, left 8, right 8, full 9. Why?

## Part 5 · Visualization

### Load the Iris measurements

The figure should answer a question already defined by the table. We revisit Iris from Week 3.

In [ ]:
raw = load_iris(as_frame=True)
d = raw.data.copy()
d.columns = ["Sepal.Length", "Sepal.Width",
             "Petal.Length", "Petal.Width"]
d["Species"] = pd.Categorical.from_codes(
    raw.target, raw.target_names)
print(len(d), d["Sepal.Width"].mean())
# 150, 3.057333 cm

> 150 flowers; mean sepal width 3.057333 cm.

### Save the scatter plot

Select the output file device explicitly, draw, then close it.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 16 / 3))
ax.scatter(d["Sepal.Length"], d["Sepal.Width"],
           facecolors="none",
           edgecolors="black")
ax.axhline(d["Sepal.Width"].mean(),
           color="#A88800")
ax.set(xlabel="Sepal length (cm)",
       ylabel="Sepal width (cm)")
fig.savefig("class07-iris-base.png", dpi=150)
plt.close(fig)

In [ ]:
from IPython.display import Image
Image("class07-iris-base.png")

> The gold line is the mean width, not a fitted relationship.

### Draw one panel per species

The grammar of graphics separates data, aesthetic mappings, layers (marks plus a calculation), scales, facets and theme.

In [ ]:
colours = ["#242424", "#A88800", "#6E6E6E"]
sns.set_theme(style="whitegrid")
g = sns.relplot(
    data=d, x="Sepal.Length", y="Sepal.Width",
    hue="Species", col="Species",
    kind="scatter", palette=colours,
    height=3.5, aspect=6 / 7)
g.refline(y=d["Sepal.Width"].mean(),
          color="#A88800")
g.set_axis_labels("Sepal length (cm)",
                  "Sepal width (cm)")
g.savefig("class07-iris-facets.pdf")
plt.close(g.figure)

In [ ]:
g.figure

> Every panel uses the overall mean as the same gold reference.

## Part 6 · From analysis to a saved figure

### Check eligibility before summarizing

- Use the checked left join.
- Keep records with known hours greater than five and a known positive achievement total.
- Retain the excluded rows for inspection.

In [ ]:
checked = joined.assign(eligible=
    joined.hours.notna() & (joined.hours > 5) &
    joined.total.notna() & (joined.total > 0) &
    joined.genre.notna())
excluded = checked.loc[~checked.eligible].copy()
eligible = checked.loc[checked.eligible].copy()
print(excluded[["player", "game", "hours",
                "genre", "total"]])
print(len(plays), len(eligible),
      len(excluded))  # 8, 6, 2

> Dan–Portal: hours unknown. Dan–Tunic: no lookup, so no denominator.

### Compute the genre summary

In [ ]:
genre_summary = (eligible
    .assign(pct=lambda d:
            100 * d.achieved / d.total)
    .groupby("genre", as_index=False)
    .agg(records=("player", "size"),
         known_hours=("hours", "sum"),
         pct=("pct", "mean"))
    .sort_values("pct", ascending=False))
print(genre_summary)

> Each eligible record has equal weight; each genre has two records.

### Save the genre figure

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(genre_summary.genre,
              genre_summary.pct,
              color="#FFD400",
              edgecolor="#242424")
ax.bar_label(bars, fmt="%.1f")
ax.set(xlabel="Genre",
       ylabel="Mean achievement completion (%)",
       ylim=(0, 50))
fig.tight_layout()
fig.savefig("class07-genre-summary.pdf")
plt.close(fig)

In [ ]:
fig

> puzzle 43.3, platformer 41.7, roguelike 33.7.

### Save the summary and the exclusions

The CSV, the chart and the exclusion record are the inputs for Class B's report.

In [ ]:
genre_summary.to_csv(
    "class07-genre-summary.csv", index=False)
excluded.to_csv("class07-excluded.csv",
                index=False)

### Recap

- Say what each row represents and how many values a summary uses.
- Keep the original data and a record of exclusions.
- Validate lookup keys; tell unmatched rows from duplicated matches.
- Save a labelled figure with its numerical summary and the code that made both.